# Phenotype ↔ Disease Relation-Wise Merge

Merges Phenotype–Disease triples from PrimeKG; resolves disease tail names via DO;
deduplicates by `(head, relation, tail)`; and saves the result.



## 0. Configuration

In [1]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PHENOTYPE_DISEASE/ALL_PHENOTYPE_DISEASE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Disease Lookup Dictionary

In [2]:
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


## 2. Load KG Sources

### 2a. PrimeKG

In [9]:
PrimeKG_Phenotype_Disease = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Phenotype_Disease_1.csv')
PrimeKG_Phenotype_Disease.columns = PrimeKG_Phenotype_Disease.columns.str.lower()
PrimeKG_Phenotype_Disease['tail_detail_name'] = PrimeKG_Phenotype_Disease['tail'].map(DOID_disname_dict)
print(f"PrimeKG: {len(PrimeKG_Phenotype_Disease):,} rows | columns: {list(PrimeKG_Phenotype_Disease.columns)}")

PrimeKG_Phenotype_Disease['kg_type'] = 'Generalised'
PrimeKG_Phenotype_Disease['species'] = 'Homo species'

PrimeKG_Phenotype_Disease.head(2)

PrimeKG: 51,887 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000135,Phenotype_Disease,DOID:0090070,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID_ID,Hypogonadism,hypogonadotropic hypogonadism,Generalised,Homo species
1,HP:0010299,Phenotype_Disease,DOID:4154,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID_ID,Abnormality of dentin,dentinogenesis imperfecta,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [10]:
SOURCE_DFS = [('PrimeKG_Phenotype_Disease', PrimeKG_Phenotype_Disease)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[PrimeKG_Phenotype_Disease] ✓ no duplicates


In [11]:
PrimeKG_Phenotype_Disease = PrimeKG_Phenotype_Disease.loc[:, ~PrimeKG_Phenotype_Disease.columns.duplicated(keep='first')]
print("[PrimeKG_Phenotype_Disease] ✓ clean")

[PrimeKG_Phenotype_Disease] ✓ clean


## 4. Align Schemas and Concatenate

In [12]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 51,887 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000135,Phenotype_Disease,DOID:0090070,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID_ID,Hypogonadism,hypogonadotropic hypogonadism,Generalised,Homo species
1,HP:0010299,Phenotype_Disease,DOID:4154,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID_ID,Abnormality of dentin,dentinogenesis imperfecta,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [13]:
final_df['head_type']  = 'Phenotype'
final_df['tail_type']  = 'Disease'
final_df['relation']   = 'Phenotype_Disease'
final_df['head_id_is'] = 'HPO'
final_df['tail_id_is'] = 'DOID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Phenotype_Disease'}
Unique kg_source: {'PrimeKG'}


## 6. Deduplicate and Add Schema Columns

In [14]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 51,887 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000002,Phenotype_Disease,DOID:0090070,Phenotype,phenotype present,Disease,PrimeKG,HPO,DOID,Abnormality of body height,hypogonadotropic hypogonadism,Generalised,Homo species
1,HP:0000003,Phenotype_Disease,DOID:0050691,Phenotype,phenotype present,Disease,PrimeKG,HPO,DOID,Multicystic kidney dysplasia,branchiooculofacial syndrome,Generalised,Homo species
2,HP:0000003,Phenotype_Disease,DOID:0050777,Phenotype,phenotype present,Disease,PrimeKG,HPO,DOID,Multicystic kidney dysplasia,Joubert syndrome,Generalised,Homo species
3,HP:0000003,Phenotype_Disease,DOID:0050778,Phenotype,phenotype present,Disease,PrimeKG,HPO,DOID,Multicystic kidney dysplasia,Meckel syndrome,Generalised,Homo species
4,HP:0000003,Phenotype_Disease,DOID:0050887,Phenotype,phenotype present,Disease,PrimeKG,HPO,DOID,Multicystic kidney dysplasia,Townes-Brocks syndrome,Generalised,Homo species


## 7. QC — NaN Check

In [15]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 8. Save Output

In [18]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 51,887 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_DISEASE/ALL_PHENOTYPE_DISEASE.csv


# Final

In [2]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [3]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='tail')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,HP:0000002,Phenotype_Disease,DOID:0090070,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Abnormality of body height,hypogonadotropic hypogonadism,Generalised,Homo species,Homo sapiens,Homo sapiens
1,HP:0000003,Phenotype_Disease,DOID:0050691,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Multicystic kidney dysplasia,branchiooculofacial syndrome,Generalised,Homo species,Homo sapiens,Homo sapiens
2,HP:0000003,Phenotype_Disease,DOID:0050777,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Multicystic kidney dysplasia,Joubert syndrome,Generalised,Homo species,Homo sapiens,Homo sapiens
3,HP:0000003,Phenotype_Disease,DOID:0050778,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Multicystic kidney dysplasia,Meckel syndrome,Generalised,Homo species,Homo sapiens,Homo sapiens
4,HP:0000003,Phenotype_Disease,DOID:0050887,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Multicystic kidney dysplasia,Townes-Brocks syndrome,Generalised,Homo species,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51882,HP:3000047,Phenotype_Disease,DOID:0050175,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Abnormal glossopharyngeal nerve morphology,tick-borne encephalitis,Generalised,Homo species,Homo sapiens,Homo sapiens
51883,HP:3000047,Phenotype_Disease,DOID:0080918,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Abnormal glossopharyngeal nerve morphology,polymicrogyria,Generalised,Homo species,Homo sapiens,Homo sapiens
51884,HP:3000047,Phenotype_Disease,DOID:14423,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Abnormal glossopharyngeal nerve morphology,glossopharyngeal neuralgia,Generalised,Homo species,Homo sapiens,Homo sapiens
51885,HP:3000050,Phenotype_Disease,DOID:3322,Phenotype,phenotype present,Disease,PrimeKG,HPO_ID,DOID,Abnormality of odontoid tissue,GM1 gangliosidosis,Generalised,Homo species,Homo sapiens,Homo sapiens


In [4]:
final_file[final_file['tail'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species


In [5]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 51,887 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_DISEASE/ALL_PHENOTYPE_DISEASE.csv
